In [ ]:
!pip install -q google-genai pandas

In [ ]:
!ls -la /content/drive

ls: cannot access '/content/drive': No such file or directory


In [ ]:
!rm -rf /content/drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# =========================
# 2) Imports & Setup
# =========================
from pathlib import Path
import pandas as pd
import json
import time
from google import genai
from google.genai import errors
from google.colab import userdata

In [ ]:
# =========================
# 3) Drive Paths
# =========================
BASE_DIR = Path("/content/drive/MyDrive/Digital_Speech")

TRANSCRIPTS_DIR = BASE_DIR / "transcripts_corrected"
OUTPUT_DIR = BASE_DIR / "skills_extraction"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = OUTPUT_DIR / "video_extracted_skills1.csv"

print("Transcripts folder exists:", TRANSCRIPTS_DIR.exists())
print("Output folder:", OUTPUT_DIR)

Transcripts folder exists: True
Output folder: /content/drive/MyDrive/Digital_Speech/skills_extraction


In [ ]:


from google import genai

API_KEY = "AQ.Ab8RN6IFNjY1PEebJQxV1R5YdP0W2wEu5ID41EaUCOwEl--odA"

client = genai.Client(api_key=API_KEY)

In [ ]:
# =========================
# 5) Load Already Processed Files
# =========================
if OUTPUT_CSV.exists():
    existing_df = pd.read_csv(OUTPUT_CSV)

    print("Existing columns:", existing_df.columns.tolist())

    if "file_name" in existing_df.columns:
        processed_files = set(existing_df["file_name"].dropna().unique())
    else:
        processed_files = set()
        existing_df["file_name"] = ""

    all_results = existing_df.to_dict("records")

else:
    processed_files = set()
    all_results = []

print("Already processed files:", len(processed_files))

Existing columns: ['file_name', 'unit_number', 'unit_title', 'lesson_number', 'lesson_title', 'skill', 'concept', 'learning_objective', 'difficulty_level', 'evidence_from_transcript']
Already processed files: 62


In [ ]:
# =========================
# 6) Skill Extraction Function
# =========================
def extract_skills_from_transcript(transcript_text, file_name):
    prompt = f"""
You are an expert in Arabic educational content analysis and mathematics curriculum alignment.

Analyze the following Arabic transcript from a Grade 3 mathematics educational video.

Extract the learning skills and concepts mentioned in the lesson.

Return ONLY valid JSON list. Do not add explanations.

Each item must follow this structure:

[
  {{
    "file_name": "{file_name}",
    "unit_number": "",
    "unit_title": "",
    "lesson_number": "",
    "lesson_title": "",
    "skill": "",
    "concept": "",
    "learning_objective": "",
    "difficulty_level": "Easy/Medium/Hard",
    "evidence_from_transcript": ""
  }}
]

Rules:
- Use Arabic for skill, concept, learning_objective, and evidence.
- Try to infer unit number, lesson number, and lesson title from the file name and transcript.
- If something is not available, write "NA".
- Do not return markdown.
- Do not wrap JSON in ```json.
- Return valid JSON only.

File name:
{file_name}

Transcript:
{transcript_text}
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    text = response.text.strip()

    if text.startswith("```json"):
        text = text.replace("```json", "").replace("```", "").strip()
    elif text.startswith("```"):
        text = text.replace("```", "").strip()

    return json.loads(text)

In [ ]:
# =========================
# 7) Safe Retry Function
# =========================
def safe_extract(transcript_text, file_name, max_retries=5):
    for attempt in range(max_retries):
        try:
            return extract_skills_from_transcript(
                transcript_text=transcript_text,
                file_name=file_name
            )

        except errors.ServerError:
            wait = 30 * (attempt + 1)
            print(f"Server busy 503. Retry {attempt+1}/{max_retries} after {wait} seconds...")
            time.sleep(wait)

        except json.JSONDecodeError:
            print(f"JSON parsing error in file: {file_name}")
            return None

        except Exception as e:
            print(f"Error in file {file_name}: {e}")
            return None

    print(f"Failed after retries: {file_name}")
    return None

In [ ]:
# =========================
# 8) Process Transcript Files
# =========================
txt_files = sorted(TRANSCRIPTS_DIR.glob("*.txt"))

print("Total txt files:", len(txt_files))

for txt_file in txt_files:
    if txt_file.name in processed_files:
        print(f"Skipping already processed: {txt_file.name}")
        continue

    print("\nProcessing:", txt_file.name)

    transcript_text = txt_file.read_text(encoding="utf-8", errors="ignore")

    if len(transcript_text.strip()) < 50:
        print("Skipped: transcript too short")
        continue

    extracted_data = safe_extract(
        transcript_text=transcript_text,
        file_name=txt_file.name
    )

    if extracted_data is None:
        continue

    if isinstance(extracted_data, dict):
        extracted_data = [extracted_data]

    for item in extracted_data:
        item["file_name"] = txt_file.name
        all_results.append(item)

    df = pd.DataFrame(all_results)
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

    print("Saved current results to:", OUTPUT_CSV)

    time.sleep(20)

print("\nFinished.")
print("Final CSV:", OUTPUT_CSV)

Total txt files: 74
Skipping already processed: NA - -2021- الرياضيات ｜ الوحدة 1 ｜ التقريب ｜ الدرس 4 ｜ الصف 3 ｜ الفصل1.txt
Skipping already processed: NA - -2021- الرياضيات ｜ الوحدة 1 ｜ القيمة المنزلية ｜ الدرس 2 ｜ الصف 3 ｜ الفصل 1.txt
Skipping already processed: NA - -2021- الرياضيات ｜ الوحدة 1 ｜ المقارنة بين الاعداد ضمن 9999 ｜ الدرس 3 ｜ الصف 3 ｜ الفصل1.txt
Skipping already processed: NA - -2021- الرياضيات ｜ الوحدة 1 ｜ جمع عددين 9999 دون حمل ｜ الدرس 5 ｜ الصف 3 ｜ الفصل 1.txt
Skipping already processed: NA - -2021- الرياضيات ｜ الوحدة1 ｜ المقارنة بين الاعداد ضمن9999 ｜ الدرس3 ｜ الصف3 ｜الفصل1.txt
Skipping already processed: NA - -2021- درس 6 ｜ الوحدة 1 ｜ جمع عددين ضمن9999مع حمل ｜ الصف 3 ｜ الفصل 1 ｜ الرياضيات.txt
Skipping already processed: NA - -2021- درس 7 ｜ الوحدة 1 ｜ طرح عددين ضمن 9999 دون استلاف ｜ الصف 3 ｜ الفصل 1 ｜ الرياضيات.txt
Skipping already processed: NA - -2021- درس 7 ｜ الوحدة 2 ｜ المربع ｜ الصف 3 ｜ الفصل 1 ｜ الرياضيات.txt
Skipping already processed: NA - -2021- درس 8 ｜ الوحدة 1 ｜

In [ ]:
print(TRANSCRIPTS_DIR)
print(TRANSCRIPTS_DIR.exists())

/content/drive/MyDrive/Digital_Speech/transcripts_corrected
True


In [ ]:
!find "/content/drive/MyDrive/Digital_Speech" -type d

/content/drive/MyDrive/Digital_Speech
/content/drive/MyDrive/Digital_Speech/transcripts_raw
/content/drive/MyDrive/Digital_Speech/math_audio
/content/drive/MyDrive/Digital_Speech/transcripts_corrected
/content/drive/MyDrive/Digital_Speech/skills_extraction


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
